<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 16 - Analítica de Big Data en Ecosistemas de Almacenamiento

**Módulo:** Big Data Aplicado (BDA - 5075)

**Contenido:** 1.3 Analítica de Big Data en ecosistemas de almacenamiento — SQL sobre Data Lakes, Spark SQL, catálogos de datos y optimización de consultas distribuidas

---

## Objetivos
- Integrar herramientas analíticas con sistemas de almacenamiento
- Ejecutar SQL sobre Data Lakes con Spark SQL
- Comprender los catálogos de datos (Hive Metastore, AWS Glue, Unity Catalog)
- Optimizar consultas en entornos distribuidos (particionado, caché, broadcast)

In [ ]:
# Instalar PySpark
!pip install pyspark -q

## SQL sobre Data Lakes

### ¿Qué es la analítica sobre Data Lakes?

Los Data Lakes modernos permiten ejecutar **SQL directamente sobre archivos** (Parquet, ORC, JSON) sin necesidad de moverlos a una base de datos tradicional.

| Servicio | Proveedor | Motor SQL | Descripción |
|----------|-----------|-----------|-------------|
| **Athena** | AWS | Presto/Trino | SQL serverless sobre S3 |
| **Synapse** | Azure | T-SQL distribuido | Analytics sobre Azure Data Lake |
| **BigQuery** | GCP | Dremel | SQL columnar a escala masiva |
| **Spark SQL** | Open Source | Catalyst Optimizer | SQL sobre cualquier fuente de datos |

**Ventajas clave:**
- Schema-on-read: el esquema se define al consultar, no al almacenar
- No mover datos: el análisis va a los datos, no al revés
- Coste por consulta: sin infraestructura fija mantenida

### Catálogo de Datos

Un **catálogo de datos** es el inventario centralizado de metadatos del Data Lake:
- Esquemas de tablas y tipos de datos
- Ubicaciones físicas de los archivos
- Estadísticas para optimización de consultas
- Linaje de datos y permisos

**Herramientas:** Apache Hive Metastore, AWS Glue Data Catalog, Unity Catalog (Databricks)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, month, desc

spark = SparkSession.builder \
    .appName("AnalíticaDataLake") \
    .master("local[*]") \
    .getOrCreate()

# ========================================
# CREAR DATOS DE EJEMPLO (simula un Data Lake)
# ========================================
import random
from datetime import datetime, timedelta

def generar_ventas(n):
    productos = {"Laptop": ("Electrónica", 999.99), "Smartphone": ("Electrónica", 699.99),
                 "Camiseta": ("Ropa", 29.99), "Silla": ("Hogar", 149.99), "Mesa": ("Hogar", 299.99)}
    regiones = ["Norte", "Sur", "Este", "Oeste"]
    base = datetime(2024, 1, 1)
    data = []
    for i in range(n):
        prod = random.choice(list(productos.keys()))
        cat, precio = productos[prod]
        fecha = (base + timedelta(days=random.randint(0, 364))).strftime("%Y-%m-%d")
        data.append((i+1, fecha, prod, cat, round(precio * random.uniform(0.9, 1.1), 2),
                     random.randint(1, 5), random.choice(regiones)))
    return data

schema_cols = ["venta_id", "fecha", "producto", "categoria", "precio", "cantidad", "region"]
df_ventas = spark.createDataFrame(generar_ventas(10000), schema_cols)

# Simular escritura en Data Lake (Parquet particionado)
df_ventas.write.mode("overwrite").partitionBy("categoria").parquet("/tmp/data_lake_ventas")
print("✅ Data Lake simulado creado en /tmp/data_lake_ventas")
print(f"   Registros: {df_ventas.count()}")

# ========================================
# SPARK SQL SOBRE EL DATA LAKE
# ========================================

# Leer desde el Data Lake
df_lake = spark.read.parquet("/tmp/data_lake_ventas")

# Registrar como vista temporal (simula catálogo de datos)
df_lake.createOrReplaceTempView("ventas_lake")
print("\nEsquema del Data Lake:")
df_lake.printSchema()

# --- Consulta 1: Ventas totales por categoría ---
print("\n=== Ventas totales por categoría ===")
spark.sql("""
    SELECT 
        categoria,
        COUNT(*) as num_ventas,
        SUM(precio * cantidad) as total_ventas,
        AVG(precio) as precio_medio
    FROM ventas_lake
    GROUP BY categoria
    ORDER BY total_ventas DESC
""").show()

# --- Consulta 2: Top 5 productos con ventas sobre la media ---
print("=== Productos con ventas sobre la media ===")
spark.sql("""
    SELECT producto, ROUND(SUM(precio * cantidad), 2) as total
    FROM ventas_lake
    GROUP BY producto
    HAVING SUM(precio * cantidad) > (SELECT AVG(precio * cantidad) FROM ventas_lake)
    ORDER BY total DESC
""").show()

# --- Consulta 3: Análisis de crecimiento mensual con Window Functions ---
print("=== Ranking de regiones por mes ===")
spark.sql("""
    SELECT 
        SUBSTR(fecha, 1, 7) as mes,
        region,
        ROUND(SUM(precio * cantidad), 2) as total,
        RANK() OVER (PARTITION BY SUBSTR(fecha, 1, 7) ORDER BY SUM(precio * cantidad) DESC) as ranking
    FROM ventas_lake
    GROUP BY SUBSTR(fecha, 1, 7), region
    ORDER BY mes, ranking
    LIMIT 20
""").show()

print("\n✅ Ejemplo de SQL sobre Data Lake completado")

---

## Optimización de Consultas Distribuidas

Ahora que sabemos consultar un Data Lake con Spark SQL, veamos las técnicas para **optimizar el rendimiento** en entornos distribuidos.

## 1. Configuración y datos de ejemplo

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast
import time

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("OptimizacionSpark") \
    .master("local[*]") \
    .getOrCreate()

# Crear datos de ejemplo
ventas_data = [
    (1, "2024-01-15", "Laptop", "Electrónica", 999.99, 2, 101),
    (2, "2024-01-15", "Mouse", "Electrónica", 29.99, 5, 102),
    (3, "2024-01-16", "Camiseta", "Ropa", 19.99, 3, 101),
    (4, "2024-01-16", "Laptop", "Electrónica", 999.99, 1, 103),
    (5, "2024-01-17", "Pantalón", "Ropa", 49.99, 2, 102),
]

clientes_data = [
    (101, "Ana García", "Madrid"),
    (102, "Luis Pérez", "Barcelona"),
    (103, "María López", "Sevilla"),
]

df_ventas = spark.createDataFrame(ventas_data, 
    ["venta_id", "fecha", "producto", "categoria", "precio", "cantidad", "cliente_id"])

df_clientes = spark.createDataFrame(clientes_data,
    ["cliente_id", "nombre", "ciudad"])

print("Datos de ventas:")
df_ventas.show()

## 2. Particionado

In [ ]:
print("=== PARTICIONADO ===")

# Escribir particionado por fecha
df_ventas.write \
    .mode("overwrite") \
    .partitionBy("fecha") \
    .parquet("/tmp/ventas_por_fecha")

print("Datos guardados particionados por fecha")

# Leer con filtro (Spark optimiza automáticamente)
df_filtrado = spark.read.parquet("/tmp/ventas_por_fecha") \
    .filter("fecha = '2024-01-15'")

print("\nDatos filtrados por fecha '2024-01-15':")
df_filtrado.show()

# Ver plan de ejecución
print("\nPlan de ejecución (debería mostrar partition pruning):")
df_filtrado.explain(True)

## 3. Caché de datos

In [ ]:
print("=== CACHÉ DE DATOS ===")

# Cachear DataFrame frecuentemente usado
df_ventas.cache()

# Primera consulta (carga en caché)
df_ventas.count()

# Consultas posteriores (desde caché)
inicio = time.time()
for _ in range(10):
    df_ventas.filter(col("categoria") == "Electrónica").count()
print(f"Tiempo con caché (10 consultas): {time.time() - inicio:.4f}s")

# Liberar caché
df_ventas.unpersist()
print("Caché liberado")

## 4. Broadcast Join

In [ ]:
print("=== BROADCAST JOIN ===")

# Para tablas pequeñas, usar broadcast
df_resultado = df_ventas.join(
    broadcast(df_clientes),  # Envía tabla pequeña a todos los workers
    "cliente_id"
)

print("Resultado del JOIN con broadcast:")
df_resultado.show()

print("\nPlan de ejecución (debería mostrar BroadcastHashJoin):")
df_resultado.explain()

## 5. Repartition vs Coalesce

In [ ]:
print("=== COALESCE vs REPARTITION ===")

# Ver particiones actuales
print(f"Particiones originales: {df_ventas.rdd.getNumPartitions()}")

# Coalesce: reduce particiones (shuffle mínimo)
df_coalesce = df_ventas.coalesce(1)
print(f"Después de coalesce(1): {df_coalesce.rdd.getNumPartitions()}")

# Repartition: redistribuye (shuffle completo, mejor balance)
df_repartition = df_ventas.repartition(4)
print(f"Después de repartition(4): {df_repartition.rdd.getNumPartitions()}")

# Repartition por columna (optimiza joins posteriores)
df_por_categoria = df_ventas.repartition("categoria")
print(f"Reparticionado por categoria: {df_por_categoria.rdd.getNumPartitions()}")

## Buenas prácticas de optimización

| Práctica | Cuándo usar | Impacto |
|----------|-------------|--------|
| **Particionado** | Consultas filtradas frecuentes | 🔥🔥🔥 Alto |
| **Caché** | DataFrames reutilizados | 🔥🔥 Medio-Alto |
| **Broadcast** | Joins con tabla pequeña | 🔥🔥 Medio |
| **Filtrar temprano** | Siempre | 🔥🔥🔥 Alto |
| **Seleccionar columnas** | Datos anchos | 🔥🔥 Medio |

In [ ]:
# Cerrar sesión
spark.stop()
print("\n✅ Sesión de Spark cerrada")

---

## Recursos adicionales

- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [Tuning Spark](https://spark.apache.org/docs/latest/tuning.html)